<a href="https://colab.research.google.com/github/estefania-apaza/state-capacity-protest-peru/blob/main/Avance_de_base_de_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Avance de la Base de Datos
Propuesta de Investigación
- Curso: Estadística para el Análisis Político 2
- Nombres: Estefanía Apaza (20230487) y Diego Luyo (20230934)


Insumos

[Libro de Códigos](https://docs.google.com/spreadsheets/d/1U5zJE58q_83H3Lc0-il9jx8QrDb9EX78/edit?gid=266433641#gid=266433641)

[Base de Eventos de Protesta](https://github.com/estefania-apaza/state-capacity-protest-peru/blob/main/Base%20de%20Eventos%20de%20Protesta%20del%20Peru%CC%81_1980_2025.csv) (Aragón, et al.) - Necesario subirla a Google Colab para correr el código

### Avance de la variable Y


**1. Limpieza y filtrado de la base de datos**

In [30]:
import pandas as pd

# Cargado de la base de la Escuela de Gobierno y Políticas Públicas PUCP
file_path = "Base de Eventos de Protesta del Perú_1980_2025.csv"
df_egpp = pd.read_csv(file_path, sep=';', encoding='latin-1', on_bad_lines='skip')

# Limpieza de nombres de columnas
df_egpp.columns = df_egpp.columns.str.strip().str.lower()
df_egpp.columns = df_egpp.columns.str.replace('ñ', 'n').str.replace('á', 'a').str.replace('é', 'e').str.replace('í', 'i').str.replace('ó', 'o').str.replace('ú', 'u')

# Selección preliminar de columnas
columnas_analisis = [
    'id', 'ano', 'mes_id', 'presidente_id',
    'region_id', 'provincia_id', 'distrito_id',
    'sector_id_1', 'actor_1_id', 'sector_id_2', 'actor_2_id',
    'accion_1_id', 'accion_2_id', 'accion_3_id', 'accion_4_id',
    'duracion_horas', 'numero_participantes',
    'numero_heridos', 'numero_muertos', 'numero_detenidos',
    'adversario_id', 'institucion_id',
    'reclamo_id', 'sub_reclamo_id']

# Creación de la base final
df_final = df_egpp[[c for c in columnas_analisis if c in df_egpp.columns]].copy()

df_final.columns
df_final.head(5)

,id,ano,mes_id,presidente_id,region_id,provincia_id,distrito_id,sector_id_1,actor_1_id,sector_id_2,...,accion_4_id,duracion_horas,numero_participantes,numero_heridos,numero_muertos,numero_detenidos,adversario_id,institucion_id,reclamo_id,sub_reclamo_id
0,1,1980,1,1,15,1501,150101,18,102,NaN,...,NaN,24.0,NaN,NaN,NaN,NaN,9,909.0,1,105
1,2,1980,1,1,15,1501,150101,13,104,NaN,...,NaN,24.0,2500.0,NaN,NaN,NaN,7,712.0,1,101
2,3,1980,1,1,15,1501,150108,103,206,NaN,...,NaN,24.0,NaN,15.0,NaN,NaN,1,105.0,4,407
3,4,1980,1,1,15,1501,150101,26,127,NaN,...,NaN,24.0,NaN,NaN,NaN,NaN,9,909.0,1,104
4,5,1980,1,1,8,801,80101,21,119,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,5,508.0,1,109


**2. Definición del umbral de la violencia para la variable dependiente: "Protesta violenta"**

Criterios

*   Alguna de las acciones implica violencia según el Libro de Códigos
*   Existe alguna víctima física



In [31]:
def variable_violencia(row):
    # Códigos de Acción seleccionados por implicar violencia según el Libro de Códigos (107-114, 119, 123)
    codigo_violencia = [107, 108, 109, 110, 111, 112, 113, 114, 119, 123]

    # Extraemos los IDs mencionados de las columnas de Acción (1-4)
    acciones = [row.get('accion_1_id'), row.get('accion_2_id'),
                row.get('accion_3_id'), row.get('accion_4_id')]

    # Criterio 1 - Acción implica violencia
    es_accion_violenta = any(acc in codigo_violencia for acc in acciones)

    # Criterio 2 - Existen víctimas físicas
    muertos = pd.to_numeric(row.get('numero_muertos', 0), errors='coerce') or 0
    heridos = pd.to_numeric(row.get('numero_heridos', 0), errors='coerce') or 0
    tiene_victimas = (muertos > 0 or heridos > 0)

    if es_accion_violenta or tiene_victimas:
        return 1
    return 0

# Creamos la variable dependiente con la función
df_final['violencia_y'] = df_final.apply(variable_violencia, axis=1)

print(df_final['violencia_y'].value_counts())

violencia_y
0    21144
1     3882
Name: count, dtype: int64


**3. Revisión de la base**

In [32]:
print("Descripción del avance de la Base de Datos")
print(f"Columnas: {len(df_final.columns)}")
print(f"Observaciones: {len(df_final)}")
print("\nConteo de la Variable Dependiente (Protesta violenta):")
print(df_final['violencia_y'].value_counts())

Descripción del avance de la Base de Datos
Columnas: 25
Observaciones: 25026

Conteo de la Variable Dependiente (Protesta violenta):
violencia_y
0    21144
1     3882
Name: count, dtype: int64


In [33]:
# Descarga del avance
from google.colab import files

df_final.to_csv('base_protestas_procesada.csv', index=False, encoding='utf-8-sig')
files.download('base_protestas_procesada.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Referencias**

Aragón, Jorge, Moisés Arce, Renzo Aurazo y Omar Coronel. 2025. Base de Eventos de Protestas del Perú, Versión Agosto 2025. Lima: Pontificia Universidad Católica del Perú-PUCP

Aragón, Jorge, Moisés Arce, Renzo Aurazo y Omar Coronel. 2025. Base de Eventos de Protestas del Perú: Libro de Códigos, Versión Agosto 2025. Lima: Pontificia Universidad Católica del Perú-PUCP

### Avance de la variable X1 - Autoridad estatal

**1. Elaboración de la función para la variable independiente 1: "Autoridad Estatal"**

* Proxy de Sullivan: Reportes de crímenes
* Propuesta: Acceso a la justicia

1. "En los últimos 12 meses, ¿ha tenido algún desacuerdo, conflicto o problema...?"

2. "Por este desacuerdo o conflicto acudió ud. a:"

* Uso del Estado: 7-12
* No uso del Estado: 1-6


In [34]:
# Cargado de el módulo de Gobernabilidad de la ENAHO 2024 como inicio
file_path = "Enaho01B-2024-2.csv"
enaho2024 = pd.read_csv(file_path, sep=',', encoding='latin-1', on_bad_lines='skip', low_memory=False)
enaho2024.columns = enaho2024.columns.str.strip().str.upper()

if 'AÃ‘O' in enaho2024.columns:
    enaho2024.rename(columns={'AÃ‘O': 'ANIO'}, inplace=True)
elif 'AÑO' in enaho2024.columns:
    enaho2024.rename(columns={'AÑO': 'ANIO'}, inplace=True)

In [35]:
def definir_autoridad(df, anio):
    # Definimos las columnas
    cols_conflictos = [f'P24_{i}' for i in range(1, 12)]
    cols_estado = [f'P26_{i}' for i in [3, 4, 5, 7, 8, 9, 10, 12]]

    # Convertimos a números porque en el CSV vienen como texto
    for c in cols_conflictos + cols_estado:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    # Identificar quién tuvo al menos un conflicto (1 = Sí)
    df_proceso = df.copy()
    df_proceso['tuvo_conflicto'] = (df_proceso[cols_conflictos] == 1).any(axis=1).astype(int)

    # Filtrar por víctimas
    df_victimas = df_proceso[df_proceso['tuvo_conflicto'] == 1].copy()

    if df_victimas.empty:
        print("Error: No se encontraron registros de conflictos. Revisa si el separador es correcto.")
        return pd.DataFrame()

    # Variable de quiénes acudieron al Estado
    df_victimas['acudio_estado'] = (df_victimas[cols_estado] == 1).any(axis=1).astype(int)

    # CORRECCIÓN DE UBIGEO: Convertir a string con ceros antes de cortar
    df_victimas['region'] = df_victimas['UBIGEO'].astype(str).str.zfill(6).str[:2]

    # Calcular porcentaje por región
    resultado = df_victimas.groupby('region')['acudio_estado'].mean().reset_index()
    resultado['ano'] = anio
    resultado.rename(columns={'acudio_estado': 'autoridad_x1'}, inplace=True)

    return resultado

# 5. EJECUTAR Y VER RESULTADO
df_x1_2024 = definir_autoridad(enaho2024, 2024)
print(df_x1_2024.head())

  region  autoridad_x1   ano
0     01      0.565217  2024
1     02      0.830986  2024
2     03      0.893617  2024
3     04      0.774194  2024
4     05      0.483871  2024


**Referencias**

Instituto Nacional de Estadística e Informática. (2024). Encuesta Nacional de Hogares (ENAHO) [Base de datos]. https://proyectos.inei.gob.pe/microdatos/